# Automated Content Tagging for Blogs
**Persevex Artificial Intelligence Internship — Project 1**

This notebook walks through the complete workflow:
1. **Dataset & Tag Taxonomy (`data/taxonomy.json`)**: Definitions, examples, alias map (`ML` -> `Machine Learning`, `py` -> `Python`, `k8s` -> `Cloud & DevOps`), HTML/Markdown boilerplate stripping, duplicate cluster grouping, temporal hold-out split, low-example niche tag reporting, and the 50-post human reviewer audit (`data/held_out_reviewer_sample.csv`).
2. **Baseline vs. TF-IDF + OvR Logistic Regression**: Comparing Keyword & Alias Matcher against TF-IDF (`1x` vs `2x` title weighting).
3. **Pretrained Sentence-Transformer Semantic Model (`all-MiniLM-L6-v2`)**: Splitting long articles into 60-word overlapping chunks and combining chunk embeddings via `0.70*Mean + 0.30*Max` pooling.
4. **Threshold Tuning & Unseen Evaluation**: Global and per-tag validation threshold tuning, Micro-F1, Macro-F1, Precision@3, Recall@3, metrics by tag frequency tier, review routing coverage, and error analysis.

In [1]:
import json
import os
import pandas as pd

with open('data/taxonomy.json') as f:
    taxonomy = json.load(f)

with open('data/dataset_summary.json') as f:
    ds_summary = json.load(f)

print('Total Posts:', ds_summary['total_posts'], '| Unique Clusters:', ds_summary['unique_clusters'])
print('Chronological Splits:', ds_summary['splits'])
print('Held-Out Reviewer Audit:', ds_summary['held_out_reviewer_audit'])

print('\n=== Tag Frequency & Reliability Audit ===')
display(pd.DataFrame(ds_summary['tag_distribution']))

Total Posts: 500 | Unique Clusters: 468
Chronological Splits: {'train': 350, 'val': 75, 'test': 75}
Held-Out Reviewer Audit: {'sample_size': 50, 'audit_file': 'data/held_out_reviewer_sample.csv', 'posts_with_missing_historical_tags': 3, 'missing_label_rate_pct': 6.0, 'note': "Historical author labels occasionally omit secondary tags (e.g., tagging 'Python' and 'API Development' but omitting 'Application Security')."}

=== Tag Frequency & Reliability Audit ===


,tag,total_posts,train_posts,val_posts,test_posts,reliable_for_evaluation,status_note
0,Python,151,98,26,27,True,Sufficient examples
1,Machine Learning,116,79,18,19,True,Sufficient examples
2,Deep Learning,63,47,9,7,True,Sufficient examples
3,API Development,99,73,15,11,True,Sufficient examples
4,Application Security,76,57,11,8,True,Sufficient examples
5,Data Engineering,90,65,11,14,True,Sufficient examples
6,Cloud & DevOps,87,62,12,13,True,Sufficient examples
7,Frontend Web Development,67,48,8,11,True,Sufficient examples
8,MLOps,60,40,9,11,True,Sufficient examples
9,Natural Language Processing,82,58,14,10,True,Sufficient examples


In [2]:
print('=== Sample of 50 Held-Out Posts Checked by Reviewer for Missing/Inconsistent Historical Labels ===')
rev_df = pd.read_csv('data/held_out_reviewer_sample.csv')
display(rev_df[rev_df['missing_or_inconsistent_flag'] == True].head(5))
display(rev_df.head(8))

=== Sample of 50 Held-Out Posts Checked by Reviewer for Missing/Inconsistent Historical Labels ===


,post_id,split,title,raw_author_tags,normalized_historical_tags,reviewer_verified_complete_tags,missing_or_inconsistent_flag,reviewer_audit_note
0,POST-1413,test,Packaging Python Libraries with pyproject.toml...,"[""Python""]","[""Python""]","[""API Development"", ""Python""]",True,Missing secondary tag 'API Development' in his...
6,POST-1404,val,Packaging Python Libraries with pyproject.toml...,"[""python3""]","[""Python""]","[""Machine Learning"", ""Python""]",True,Missing secondary tag 'Machine Learning' in hi...
30,POST-1362,val,Orchestrating Incremental ETL Pipelines with A...,"[""Data Engineering""]","[""Data Engineering""]","[""Data Engineering"", ""Machine Learning""]",True,Missing secondary tag 'Machine Learning' in hi...


,post_id,split,title,raw_author_tags,normalized_historical_tags,reviewer_verified_complete_tags,missing_or_inconsistent_flag,reviewer_audit_note
0,POST-1413,test,Packaging Python Libraries with pyproject.toml...,"[""Python""]","[""Python""]","[""API Development"", ""Python""]",True,Missing secondary tag 'API Development' in his...
1,POST-1358,val,Type-Safe Custom Hooks and Context Architectur...,"[""frontend"", ""security""]","[""Application Security"", ""Frontend Web Develop...","[""Application Security"", ""Frontend Web Develop...",False,Historical tags consistent with taxonomy after...
2,POST-1458,test,Mastering Python Generators and Async Iterator...,"[""cpython"", ""ner""]","[""Natural Language Processing"", ""Python""]","[""Natural Language Processing"", ""Python""]",False,Historical tags consistent with taxonomy after...
3,POST-1418,test,Concurrency Patterns with Python Asyncio and M...,"[""cpython""]","[""Python""]","[""Python""]",False,Historical tags consistent with taxonomy after...
4,POST-1416,test,Mastering Python Generators and Async Iterator...,"[""pythonic"", ""flask"", ""Application Security""]","[""API Development"", ""Application Security"", ""P...","[""API Development"", ""Application Security"", ""P...",False,Historical tags consistent with taxonomy after...
5,POST-1371,val,Understanding Self-Attention and Layer Normali...,"[""pytorch"", ""ner""]","[""Deep Learning"", ""Natural Language Processing""]","[""Deep Learning"", ""Natural Language Processing""]",False,Historical tags consistent with taxonomy after...
6,POST-1404,val,Packaging Python Libraries with pyproject.toml...,"[""python3""]","[""Python""]","[""Machine Learning"", ""Python""]",True,Missing secondary tag 'Machine Learning' in hi...
7,POST-1491,val,Quick ML thought,"[""Machine Learning""]","[""Machine Learning""]","[""Machine Learning""]",False,Historical tags consistent with taxonomy after...


In [3]:
with open('outputs/evaluation_metrics.json') as f:
    eval_metrics = json.load(f)

print('=== Step 1 Ablation: Title Weighting (1x vs 2x) on Validation Split ===')
print(json.dumps(eval_metrics['title_weighting_ablation'], indent=2))

print('\n=== Step 5: Unseen Test Split Model Comparison ===')
display(pd.DataFrame(eval_metrics['test_model_comparison']))

print('\n=== Metrics Grouped by Tag Frequency Tier ===')
display(pd.DataFrame(eval_metrics['frequency_tier_summary']))

print('\n=== Per-Tag Test Metrics ===')
display(pd.DataFrame(eval_metrics['per_tag_metrics']))

=== Step 1 Ablation: Title Weighting (1x vs 2x) on Validation Split ===
{
  "val_micro_f1_title_1x": 0.9611,
  "val_micro_f1_title_2x": 0.9714,
  "val_macro_f1_title_1x": 0.9716,
  "val_macro_f1_title_2x": 0.9782,
  "improvement_delta_micro": 0.0103,
  "conclusion": "Giving blog titles 2x weight improves validation Micro-F1 and Macro-F1 because article bodies often include incidental comparisons to other tools, whereas titles anchor the primary subjects."
}

=== Step 5: Unseen Test Split Model Comparison ===


,model,micro_f1,macro_f1,verified_complete_micro_f1,precision_at_3,recall_at_3,auto_tagged_coverage_pct,routed_to_review_pct,inference_ms_per_post
0,Keyword & Alias Matcher (Baseline),0.7532,0.7982,0.7601,0.5289,0.9067,86.7,13.3,0.40
1,TF-IDF + OvR LogReg (Title 1x),0.9195,0.9228,0.9398,0.5867,0.9867,86.7,13.3,0.12
2,TF-IDF + OvR LogReg (Title 2x),0.9425,0.9430,0.9624,0.5867,0.9933,86.7,13.3,0.13
3,TF-IDF + OvR LogReg (Title 2x + Per-Tag Thresh...,0.9283,0.9315,0.9481,0.5867,0.9933,86.7,13.3,0.13
4,Sentence-Transformer (all-MiniLM-L6-v2) + Chun...,0.9407,0.9460,0.9600,0.5911,1.0000,85.3,14.7,13.05



=== Metrics Grouped by Tag Frequency Tier ===


,frequency_tier,num_tags,tags_included,mean_tfidf_f1,mean_sbert_f1
0,High Frequency (>=90 posts),4,"[Python, Machine Learning, API Development, Da...",0.9364,0.9295
1,Low/Niche (<15 posts),2,"[Quantum Computing, WebAssembly]",1.0000,1.0000
2,Medium Frequency (15-89 posts),6,"[Deep Learning, Application Security, Cloud & ...",0.9054,0.9391



=== Per-Tag Test Metrics ===


,tag,total_corpus_posts,test_support,frequency_tier,tuned_threshold,tfidf_precision,tfidf_recall,tfidf_f1,semantic_sbert_f1
0,Python,151,27,High Frequency (>=90 posts),0.35,0.9630,0.9630,0.9630,0.9455
1,Machine Learning,116,19,High Frequency (>=90 posts),0.35,0.9048,1.0000,0.9500,0.9744
2,Deep Learning,63,7,Medium Frequency (15-89 posts),0.35,1.0000,0.8571,0.9231,0.9333
3,API Development,99,11,High Frequency (>=90 posts),0.55,0.8333,0.9091,0.8696,0.8696
4,Application Security,76,8,Medium Frequency (15-89 posts),0.30,0.8000,1.0000,0.8889,0.8889
5,Data Engineering,90,14,High Frequency (>=90 posts),0.60,1.0000,0.9286,0.9630,0.9286
6,Cloud & DevOps,87,13,Medium Frequency (15-89 posts),0.70,1.0000,0.7692,0.8696,0.9600
7,Frontend Web Development,67,11,Medium Frequency (15-89 posts),0.30,0.9167,1.0000,0.9565,1.0000
8,MLOps,60,11,Medium Frequency (15-89 posts),0.35,1.0000,0.9091,0.9524,0.9524
9,Natural Language Processing,82,10,Medium Frequency (15-89 posts),0.50,0.8889,0.8000,0.8421,0.9000


In [4]:
batch_df = pd.read_json('outputs/batch_tag_predictions.json')
print('=== Sample Batch Predictions Export (Post IDs, Suggested Tags, Scores, Snippets, Review Status & Final Approved Tags) ===')
display(batch_df[['post_id', 'title', 'post_type', 'word_count', 'suggested_tags', 'review_status', 'final_editor_approved_tags']].head(12))

=== Sample Batch Predictions Export (Post IDs, Suggested Tags, Scores, Snippets, Review Status & Final Approved Tags) ===


,post_id,title,post_type,word_count,suggested_tags,review_status,final_editor_approved_tags
0,POST-1408,Hyperparameter Optimization and Calibration fo...,standard,170,"[MLOps, Machine Learning, Cloud & DevOps]",Auto-Approved (High Confidence),"[MLOps, Machine Learning, Cloud & DevOps]"
1,POST-1410,Streaming Event Architectures with Apache Kafk...,standard,133,"[Data Engineering, Cloud & DevOps]",Auto-Approved (High Confidence),"[Data Engineering, Cloud & DevOps]"
2,POST-1411,Type-Safe Custom Hooks and Context Architectur...,standard,130,"[Frontend Web Development, API Development]",Auto-Approved (High Confidence),"[Frontend Web Development, API Development]"
3,POST-1412,Memory Management and Garbage Collection in CP...,standard,89,[Python],Auto-Approved (High Confidence),[Python]
4,POST-1413,Packaging Python Libraries with pyproject.toml...,long_multi_topic,248,"[Python, API Development]",Auto-Approved (High Confidence),"[API Development, Python]"
5,POST-1414,Cross-Validation Strategies for Imbalanced Cla...,standard,167,"[Machine Learning, MLOps, Cloud & DevOps]",Auto-Approved (High Confidence),"[Machine Learning, MLOps, Cloud & DevOps]"
6,POST-1415,Optimizing React Rendering Performance and Sta...,standard,83,[Frontend Web Development],Auto-Approved (High Confidence),[Frontend Web Development]
7,POST-1416,Mastering Python Generators and Async Iterator...,standard,189,"[Python, API Development, Application Security]",Auto-Approved (High Confidence),"[Python, API Development, Application Security]"
8,POST-1417,Writing Clean Pythonic Code with Type Hints an...,semantic_paraphrase,130,"[Natural Language Processing, Python]",Auto-Approved (High Confidence),"[Natural Language Processing, Python]"
9,POST-1418,Concurrency Patterns with Python Asyncio and M...,standard,92,[Python],Auto-Approved (High Confidence),[Python]
